# Notebook 3 - Parse XML into Holdings Table

this is where we actually read the XML and turn it into a proper table.

each filing has one infoTable element per holding with fields like:
- nameOfIssuer - company name
- cusip - unique identifier for the security (like a stock ticker but more standardized)
- value - dollar value of the position
- sshPrnamt - number of shares
- putCall - if its an option, whether its a call or put

one thing that was annoying - the XML uses namespaces (like ns1:infoTable instead of just infoTable) and the prefix varies between different filers. so you cant just search for "infoTable", you have to strip the namespace prefix first.

also the value field changed units in 2023. before that it was in thousands of dollars, after that its in actual dollars. so old quarters look 1000x smaller if you dont account for this.

In [ ]:
import os, sys, pathlib

# ----- COLAB SETUP -----
# Option A: with Google Drive (data stays between sessions)
# from google.colab import drive
# drive.mount("/content/drive")
# PROJECT_ROOT = pathlib.Path("/content/drive/MyDrive/13F-Analytics")

# Option B: without Drive (data gone when session ends, need to rerun)
# PROJECT_ROOT = pathlib.Path("/content/13F-Analytics")

# detects root automatically when running locally
PROJECT_ROOT = pathlib.Path("/content/13F-Analytics") if pathlib.Path("/content/13F-Analytics").exists() else pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()

os.chdir(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("working directory:", PROJECT_ROOT)

In [ ]:
from src.parser import build_raw_holdings

holdings_raw = build_raw_holdings()
holdings_raw.head(10)

In [ ]:
# how many positions per quarter
holdings_raw.groupby("quarter").agg(
    total_rows=("cusip", "size"),
    unique_issuers=("issuer", "nunique"),
    total_value_usd=("value_usd", "sum"),
).round(0)

In [ ]:
# check how complete the data is - some fields like put_call are only filled for options
holdings_raw.isna().mean().round(3).sort_values(ascending=False).to_frame("pct_missing")